# MorphoGraph-Lattice: BFDH + Wulff + hybrid crystal shapes

**Input:** a MOF CIF file. **Output:** the possible crystal faces, a surface energy for each face, and the outside crystal shape from three rules:

1. **BFDH** (geometry only): wide-spaced planes grow slowly, so they become big faces.
2. **Wulff** (energy): the lower the surface energy of a face, the bigger it is.
3. **Hybrid** (both together): `R = R_BFDH^(1-w) * R_Wulff^w`, with `w` from 0 (pure BFDH) to 1 (pure Wulff).

The face energies here come from a simple force field (UFF, unrelaxed, no electrostatics, no solvent). It is a rough baseline. If you have better energies (for example CHGNet or DFT) paste them into the **external energies** cell and the same hybrid step uses them.

No CHGNet is needed to run this notebook. It works for any crystal system.

In [ ]:
%pip install -q ase spglib scipy matplotlib numpy

## 1. Library files (written to disk so you can read them)

In [ ]:
import os
os.makedirs('mofmorph', exist_ok=True)
open('mofmorph/__init__.py','w').close()

In [ ]:
%%writefile mofmorph/structure.py
"""Read a CIF, find the true space group, return the conventional cell and its symmetry operations.

Many MOF CIFs are stored in P1 (no symmetry). Hydrogen positions are often disordered or
placed by software and can break the symmetry, so we search for symmetry both with and
without H and keep the higher-symmetry result (reported to the user).
"""
from dataclasses import dataclass, field
import warnings
import numpy as np
import spglib
from ase.io import read


@dataclass
class Crystal:
    name: str
    lattice: np.ndarray            # rows = conventional lattice vectors a, b, c (Angstrom)
    frac: np.ndarray               # fractional positions in the conventional cell
    numbers: np.ndarray            # atomic numbers
    spacegroup: str
    sg_number: int
    rotations: np.ndarray          # (n,3,3) integer, act on fractional coordinates
    translations: np.ndarray       # (n,3)
    symprec: float
    used_hydrogen: bool
    notes: list = field(default_factory=list)

    @property
    def recip(self):
        """Rows = reciprocal vectors a*, b*, c* (no 2*pi), so |h @ recip| = 1/d_hkl."""
        return np.linalg.inv(self.lattice).T

    @property
    def cell_lengths(self):
        return np.linalg.norm(self.lattice, axis=1)


def _dataset(cell, symprec):
    d = spglib.get_symmetry_dataset(cell, symprec=symprec)
    return d


def load(path, name=None, symprecs=(1e-3, 1e-2, 5e-2, 1e-1)):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        atoms = read(path)
    name = name or path
    notes = []
    candidates = []
    for use_h in (True, False):
        a = atoms if use_h else atoms[[s != "H" for s in atoms.get_chemical_symbols()]]
        cell = (a.cell[:], a.get_scaled_positions(), a.numbers)
        for sp in symprecs:
            d = _dataset(cell, sp)
            if d is None:
                continue
            candidates.append((d.number, use_h, -sp, sp, cell))
            break  # smallest tolerance that works for this H choice
    # prefer: higher space-group number at the smallest tolerance; with H if it ties
    best = max(candidates, key=lambda c: (c[0], c[1], c[2]))
    number, use_h, _, sp, cell = best
    with_h = [c for c in candidates if c[1]]
    if not use_h and with_h and with_h[0][0] < number:
        notes.append(f"With H atoms only space group #{with_h[0][0]} is found; without H #{number}. "
                     "H positions in the file break the symmetry; the framework symmetry is used.")
    L, p, num = spglib.standardize_cell(cell, to_primitive=False, no_idealize=False, symprec=sp)
    d = spglib.get_symmetry_dataset((L, p, num), symprec=sp)
    return Crystal(name=name, lattice=np.array(L), frac=np.array(p), numbers=np.array(num),
                   spacegroup=d.international, sg_number=d.number,
                   rotations=np.array(d.rotations), translations=np.array(d.translations),
                   symprec=sp, used_hydrogen=use_h, notes=notes)

In [ ]:
%%writefile mofmorph/faces.py
"""Allowed crystal faces, effective spacings and symmetry families.

Systematic-absence rule (primer, Proposition 1): reflection h is absent if the space group
contains an operation (R, t) with h R = h and h.t not an integer.
"""
from dataclasses import dataclass
from math import gcd
import itertools
import numpy as np


def is_absent(h, rotations, translations, tol=1e-6):
    h = np.asarray(h)
    for R, t in zip(rotations, translations):
        if np.array_equal(h @ R, h):
            ph = float(h @ t)
            if abs(ph - round(ph)) > tol:
                return True
    return False


@dataclass
class Face:
    hkl: tuple          # primitive direction (no common factor)
    order: int          # n: first allowed reflection is n*hkl
    d: float            # d_hkl of the primitive direction (Angstrom)
    d_eff: float        # d / n
    normal: np.ndarray  # Cartesian unit normal
    family: tuple       # canonical representative of the symmetry family


def _primitive(v):
    g = gcd(gcd(abs(v[0]), abs(v[1])), abs(v[2]))
    return tuple(int(x // g) for x in v)


def enumerate_faces(crystal, max_index=6, absence_ops=None):
    """absence_ops=(rotations, translations): operations used for the systematic-absence test.
    By default the crystal's own operations. A deformed crystal (deform.py) passes the ORIGINAL
    operations here, because absences depend only on fractional coordinates (Proposition 14),
    while its families are the orbits of the smaller isometric subgroup."""
    rots, trans = crystal.rotations, crystal.translations
    if absence_ops is not None:
        rots_abs, trans_abs = absence_ops
    else:
        rots_abs, trans_abs = rots, trans
    B = crystal.recip
    point_ops = {tuple(R.ravel()) for R in rots}
    point_ops = [np.array(r).reshape(3, 3) for r in point_ops]
    faces = {}
    rng = range(-max_index, max_index + 1)
    for v in itertools.product(rng, rng, rng):
        if v == (0, 0, 0) or _primitive(v) != v:
            continue
        h = np.array(v)
        n = 1
        while is_absent(n * h, rots_abs, trans_abs):
            n += 1
            if n > 12:
                break
        g = h @ B
        d = 1.0 / np.linalg.norm(g)
        # symmetry family: all h R (row vector convention), canonical = max tuple
        fam = max(tuple(int(x) for x in (h @ R)) for R in point_ops)
        faces[v] = Face(hkl=v, order=n, d=d, d_eff=d / n, normal=g / np.linalg.norm(g), family=fam)
    return faces


def families(faces):
    """Group faces by family; return list sorted by decreasing d_eff."""
    fam = {}
    for f in faces.values():
        fam.setdefault(f.family, []).append(f)
    out = []
    for key, fs in fam.items():
        out.append(dict(family=key, members=fs, d=fs[0].d, d_eff=fs[0].d_eff, order=fs[0].order,
                        multiplicity=len(fs), label=family_label(key, len(fs))))
    out.sort(key=lambda x: (-x["d_eff"], -x["multiplicity"]))
    return out


def label(hkl):
    return "{" + "".join(f"\\bar{{{-x}}}" if x < 0 else str(x) for x in hkl) + "}"


def plain(hkl):
    return "{" + "".join(str(abs(x)) for x in sorted(hkl, key=lambda x: -abs(x))) + "}"


def full_cubic_multiplicity(hkl):
    vals = set()
    for p in itertools.permutations(hkl):
        for sg in itertools.product((1, -1), repeat=3):
            vals.add(tuple(a * b for a, b in zip(p, sg)))
    return len(vals)


def family_label(key, multiplicity):
    """Plain label like {111}; in non-centrosymmetric tetrahedral groups (e.g. F-43m, I-43m) the
    cubic family splits in two halves, labelled {111}+ and {111}- by the sign of h*k*l."""
    lab = plain(key)
    if multiplicity < full_cubic_multiplicity(key) and all(key):
        lab += "+" if key[0] * key[1] * key[2] > 0 else "-"
    return lab

In [ ]:
%%writefile mofmorph/shape.py
"""Crystal shape as a convex polytope  P = {x : n_f . x <= r_f}  and its descriptors."""
import math
import numpy as np
from scipy.spatial import HalfspaceIntersection, ConvexHull
from scipy.special import sph_harm_y


class Shape:
    def __init__(self, normals, dists, tags=None):
        A = np.asarray(normals, float)
        A = A / np.linalg.norm(A, axis=1)[:, None]
        b = np.asarray(dists, float)
        hs = HalfspaceIntersection(np.hstack([A, -b[:, None]]), np.zeros(3))
        self.V = hs.intersections
        self.hull = ConvexHull(self.V)
        self.tags = tags if tags is not None else [None] * len(A)
        self.A, self.b = A, b
        # which input planes actually carry a face, and their areas
        self.faces = []      # list of (tag, normal, polygon vertices, area)
        groups = {}
        for eq, simp in zip(self.hull.equations, self.hull.simplices):
            key = tuple(np.round(eq[:3], 6))
            groups.setdefault(key, []).append(simp)
        for key, simps in groups.items():
            n = np.array(key)
            idx = sorted(set(np.concatenate(simps)))
            P = self.V[idx]
            c = P.mean(0)
            u = P[0] - c; u /= np.linalg.norm(u); w = np.cross(n, u)
            P = P[np.argsort(np.arctan2((P - c) @ w, (P - c) @ u))]
            area = sum(0.5 * np.linalg.norm(np.cross(self.V[s[1]] - self.V[s[0]], self.V[s[2]] - self.V[s[0]]))
                       for s in simps)
            j = int(np.argmax(A @ n))
            self.faces.append(dict(tag=self.tags[j], normal=n, poly=P, area=area))

    # ---- descriptors -------------------------------------------------------
    @property
    def volume(self):
        return self.hull.volume

    @property
    def area(self):
        return self.hull.area

    def sphericity(self):
        return math.pi ** (1 / 3) * (6 * self.volume) ** (2 / 3) / self.area

    def gyration(self):
        """Exact gyration tensor of the solid polytope by tetrahedral decomposition."""
        c0 = self.V.mean(0)
        M = 0.0; first = np.zeros(3); second = np.zeros((3, 3))
        for s in self.hull.simplices:
            T = np.vstack([c0, self.V[s]])          # tetra vertices
            vol = abs(np.linalg.det(T[1:] - T[0])) / 6
            # second moment of a tetrahedron: vol/20 * (sum_i v_i v_i^T + (sum v)(sum v)^T)
            S = T.sum(0)
            second += vol / 20 * (T.T @ T + np.outer(S, S))
            first += vol * T.mean(0); M += vol
        com = first / M
        return second / M - np.outer(com, com)

    def kappa2(self):
        lam = np.linalg.eigvalsh(self.gyration())
        return float(1.5 * np.sum(lam ** 2) / np.sum(lam) ** 2 - 0.5)

    def aspect_ratio(self):
        lam = np.sort(np.linalg.eigvalsh(self.gyration()))
        return float(math.sqrt(lam[-1] / lam[0]))

    def q_l(self, l):
        """Steinhardt-type order parameter of the area-weighted face normals.
        Separates cube (Q4=0.764, Q6=0.354) from octahedron (Q4=0.509, Q6=0.629)."""
        w = np.array([f["area"] for f in self.faces]); w = w / w.sum()
        N = np.array([f["normal"] for f in self.faces])
        theta = np.arccos(np.clip(N[:, 2], -1, 1)); phi = np.arctan2(N[:, 1], N[:, 0])
        tot = 0.0
        for m in range(-l, l + 1):
            tot += abs(np.sum(w * sph_harm_y(l, m, theta, phi))) ** 2
        return float(math.sqrt(4 * math.pi / (2 * l + 1) * tot))

    def face_fractions(self):
        out = {}
        for f in self.faces:
            out[f["tag"]] = out.get(f["tag"], 0) + f["area"]
        tot = sum(out.values())
        return {k: v / tot for k, v in out.items()}

    def summary(self):
        return dict(n_faces=len(self.faces), n_vertices=len(self.V), sphericity=self.sphericity(),
                    kappa2=self.kappa2(), aspect_ratio=self.aspect_ratio(), Q3=self.q_l(3), Q4=self.q_l(4), Q6=self.q_l(6),
                    face_area_fraction=self.face_fractions())

In [ ]:
%%writefile mofmorph/models.py
"""Growth-rate models that turn a set of faces into a shape.

All models produce the same kind of object, a polytope P = {x : n_f . x <= R_f}; they differ
only in how the face distances R_f are chosen (primer, Sections 6-8 and 14).
"""
import math
import numpy as np
from .faces import families, plain, family_label
from .shape import Shape


def _labels(faces):
    return {F["family"]: F["label"] for F in families(faces)}


def _planes(faces, rate_of_family, families_used):
    lab = _labels(faces)
    normals, dists, tags = [], [], []
    for f in faces.values():
        key = f.family
        if key in families_used:
            normals.append(f.normal); dists.append(rate_of_family[key]); tags.append(lab[key])
    return normals, dists, tags


def _match(label, rates_by_label):
    """'{111}+' matches a rate given for '{111}+' or, failing that, for '{111}'."""
    if label in rates_by_label:
        return rates_by_label[label]
    base = label.rstrip("+-")
    return rates_by_label.get(base)


def bfdh(faces, n_families=10):
    """BFDH: R_hkl proportional to 1/d_eff. Uses the n_families with largest d_eff."""
    fams = families(faces)[:n_families]
    dmax = fams[0]["d_eff"]
    rate = {F["family"]: dmax / F["d_eff"] for F in fams}       # leading family at distance 1
    n, d, t = _planes(faces, rate, set(rate))
    return Shape(n, d, t), rate


def from_rates(faces, rates_by_label):
    """rates_by_label: {'{111}': 1.0, '{100}': 1.2, ...} using plain family labels."""
    rate = {}
    for F in families(faces):
        r = _match(F["label"], rates_by_label)
        if r is not None:
            rate[F["family"]] = r
    n, d, t = _planes(faces, rate, set(rate))
    return Shape(n, d, t)


def attachment_rates(c_by_label, beta):
    """Graph rule: R_f = exp(beta * c_f), c_f = attachment number (links formed by a new node).
    beta ~ (energy per link)/(kT); beta -> 0 makes all rates equal, large beta makes the
    slowest face dominate completely."""
    return {k: math.exp(beta * c) for k, c in c_by_label.items()}


def fibonacci_sphere(n=300):
    i = np.arange(n) + 0.5
    phi = np.arccos(1 - 2 * i / n); th = math.pi * (1 + 5 ** 0.5) * i
    return np.c_[np.cos(th) * np.sin(phi), np.sin(th) * np.sin(phi), np.cos(phi)]


def grow_from_seed(faces, rates_by_label, time, r0=1.0, v_rough=None, n_sphere=400):
    """Shape at time t of a crystal grown from a spherical seed of radius r0.
    Listed faces move at their rates; all other (rough) directions move at v_rough
    (default 1.5 x the fastest listed rate), so they disappear quickly."""
    rate = {}
    for F in families(faces):
        r = _match(F["label"], rates_by_label)
        if r is not None:
            rate[F["family"]] = r
    lab = _labels(faces)
    v_rough = v_rough or 1.5 * max(rate.values())
    normals, dists, tags = [], [], []
    for f in faces.values():
        if f.family in rate:
            normals.append(f.normal); dists.append(r0 + rate[f.family] * time); tags.append(lab[f.family])
    for n in fibonacci_sphere(n_sphere):
        normals.append(n); dists.append(r0 + v_rough * time); tags.append("rough")
    return Shape(normals, dists, tags)


def describe(shape, min_frac=1e-9):
    """Name the shape from the faces whose area fraction exceeds min_frac (exact by default)."""
    fr = shape.face_fractions()
    rough = fr.get("rough", 0.0)
    if rough > 0.5:
        return "ball (mostly rough surface)"
    if rough > 0.05:
        return "rounded " + shape_name_without_rough(shape, min_frac)
    return shape_name_without_rough(shape, min_frac)


def shape_name_without_rough(shape, min_frac=1e-9):
    fr, cnt = {}, {}
    tot_all = sum(f["area"] for f in shape.faces if f["tag"] != "rough")
    keep = {}
    for f in shape.faces:
        if f["tag"] != "rough":
            keep[f["tag"]] = keep.get(f["tag"], 0) + f["area"]
    keep = {k for k, v in keep.items() if v / tot_all > min_frac}
    for f in shape.faces:
        if f["tag"] == "rough" or f["tag"] not in keep:
            continue
        fr[f["tag"]] = fr.get(f["tag"], 0) + f["area"]; cnt[f["tag"]] = cnt.get(f["tag"], 0) + 1
    tot = sum(fr.values()); fr = {k: v / tot for k, v in fr.items()}
    cnt = {k: cnt[k] for k in fr}
    # tetrahedral halves
    if set(fr) in ({"{111}+"}, {"{111}-"}) and sum(cnt.values()) == 4:
        return "tetrahedron"
    if set(fr) == {"{111}+", "{111}-"}:
        return "octahedron" if abs(fr["{111}+"] - fr["{111}-"]) < 0.02 else "truncated tetrahedron"
    if {"{111}+", "{111}-"} <= set(fr) or {"{111}+"} <= set(fr) or {"{111}-"} <= set(fr):
        merged = {}
        for k, v in fr.items():
            merged[k.rstrip("+-")] = merged.get(k.rstrip("+-"), 0) + v
        if abs(fr.get("{111}+", 0) - fr.get("{111}-", 0)) > 0.02:
            return "tetrahedral " + _basic(merged, shape, cnt).replace("truncated octahedron","trunc. octahedron")
        fr = merged
    return _basic(fr, shape, cnt)


def _basic(fr, shape, cnt):
    keys = set(fr); nv = len(shape.V)
    c = {}
    for k, v in cnt.items():
        c[k.rstrip("+-")] = c.get(k.rstrip("+-"), 0) + v
    if keys == {"{100}"} and c.get("{100}") == 6: return "cube"
    if keys == {"{111}"} and c.get("{111}") == 8: return "octahedron"
    if keys == {"{110}"} and c.get("{110}") == 12: return "rhombic dodecahedron"
    # named by face SHAPE, not by which family has more area: a {111} face that is a
    # triangle means cube corners were cut (truncated cube); a hexagon means octahedron
    # corners were cut (truncated octahedron)
    def corners(base):
        return max([len(f["poly"]) for f in shape.faces if f["tag"].rstrip("+-") == base and f["tag"] in cnt] or [0])
    if keys == {"{100}", "{111}"}:
        if nv == 12: return "cuboctahedron"
        return "truncated octahedron" if corners("{111}") >= 6 else "truncated cube"
    if keys == {"{100}", "{110}"}:
        # one combinatorial type throughout (6 squares, 12 hexagons, a chamfered cube),
        # so here the larger family names it
        return "truncated rhombic dodecahedron" if fr["{110}"] > fr["{100}"] else "chamfered cube"
    if keys == {"{100}", "{110}", "{111}"}:
        return "rhombicuboctahedron-like"
    return " + ".join(sorted(keys))

In [ ]:
%%writefile mofmorph/forcefield.py
"""Phase 4: unrelaxed cleavage (surface) energy of a crystal plane from a pairwise force field.

Model (all stated, nothing hidden):
  * van der Waals: UFF Lennard-Jones, E(r) = D [ (x/r)^12 - 2 (x/r)^6 ], Lorentz-Berthelot-like
    UFF mixing D_ij = sqrt(D_i D_j), x_ij = sqrt(x_i x_j). Pairs closer than two bonds apart
    (1-2 and 1-3) are excluded, as in UFF.
  * bonds: every bond cut by the plane costs a flat energy D_bond (UFF uses 70 kcal/mol as its
    Morse well depth). Bonds are perceived from covalent radii.
  * the cut is UNRELAXED (atoms stay where they were) and has NO electrostatics.
So gamma is an upper bound on a relaxed surface energy and, for charged frameworks such as UiO-66,
a poor absolute number; only the relative ordering of faces is the intended use.

Exact counting. For a family with plane normal n and lattice spacing d, planes sit at offsets
s + k d. A pair (i, j + T) with projections p_i, p_j is cut by a plane when s + k d lies between
them. If one unit cell of volume V holds all pairs once, planes of one-per-period have total area
V/d per cell, so   gamma(s) = (1/2) (d / V) * sum_pairs eps_pair * c_pair(s),
c = number of planes crossed, the factor 1/2 because a cut makes two surfaces.
"""
import numpy as np
from ase.data import covalent_radii, atomic_numbers
from ase.neighborlist import neighbor_list

KCAL = 4.184  # kJ per kcal
# UFF (x1 [A], D1 [kcal/mol]) Rappe et al. 1992, Table 1
UFF = {"H": (2.886, 0.044), "C": (3.851, 0.105), "N": (3.660, 0.069), "O": (3.500, 0.060),
       "Zn": (2.763, 0.124), "Zr": (3.124, 0.069), "Cu": (3.495, 0.005), "Hf": (3.141, 0.072),
       # added 2026-10-14 for CoRE MOF structures (same table). The vdW part is 1-9 % of gamma here,
       # so these enter the face energies only weakly.
       "Co": (2.872, 0.014), "Ni": (2.834, 0.015), "Cd": (2.848, 0.228), "S": (4.035, 0.274),
       "In": (4.463, 0.599), "Mn": (2.961, 0.013), "Cl": (3.947, 0.227)}
D_BOND = 70.0  # kcal/mol, UFF default Morse well depth


def pair_list(atoms, rcut=12.0, bond_scale=1.15):
    """All unique pairs (i<j images counted once) with distance, direction vector, bonded flag,
    and 'excluded' flag (1-2 or 1-3)."""
    i, j, d, D = neighbor_list("ijdD", atoms, rcut)
    cov = covalent_radii[atoms.numbers]
    bonded = d < bond_scale * (cov[i] + cov[j])
    n = len(atoms)
    # 1-2 / 1-3 exclusion works on the image-resolved graph: use (atom, image) pairs via supercell trick
    # Simpler and exact for these cells: bonds are short, so exclusion = bonded or sharing a bonded neighbour.
    # a 1-3 pair is two bonds apart; with images, check by position: pair (i, j) with vector D is 1-3 when
    # some k bonded to both with consistent vectors. Do it with explicit neighbour vectors.
    nb = {}
    for a, b, v in zip(i[bonded], j[bonded], D[bonded]):
        nb.setdefault(a, []).append((b, v))
    excl = bonded.copy()
    # vectorised 1-3: for every atom a and every pair of its bonds (b1,v1),(b2,v2), the pair (b1,b2) with vector v2-v1
    one3 = {}
    for a, lst in nb.items():
        for (b1, v1) in lst:
            for (b2, v2) in lst:
                if b1 == b2 and np.allclose(v1, v2):
                    continue
                one3.setdefault((b1, b2), []).append(v2 - v1)
    for idx, (a, b, v) in enumerate(zip(i, j, D)):
        if excl[idx]:
            continue
        for w in one3.get((a, b), ()):
            if np.allclose(v, w, atol=1e-3):
                excl[idx] = True
                break
    keep = i < j
    # pairs with i == j (an atom and its own periodic image) appear twice with +-D; keep one of them
    same = (i == j)
    first = same & (D @ np.array([1.0, 1.0e-3, 1.0e-6]) > 0)
    keep = keep | first
    return i[keep], j[keep], d[keep], D[keep], bonded[keep], excl[keep]


def pair_energies(atoms, i, j, d, bonded, excl, d_bond=D_BOND):
    """Cost (kcal/mol) of cutting each pair: -E_LJ for non-excluded non-bonded pairs, d_bond for bonds."""
    sym = np.array(atoms.get_chemical_symbols())
    x = np.array([UFF[s][0] for s in sym]); Dw = np.array([UFF[s][1] for s in sym])
    xij = np.sqrt(x[i] * x[j]); Dij = np.sqrt(Dw[i] * Dw[j])
    lj = Dij * ((xij / d) ** 12 - 2 * (xij / d) ** 6)
    cost = np.where(bonded, d_bond, np.where(excl, 0.0, -lj))
    cost = np.where((~bonded) & (~excl) & (d < 0.8 * xij), 0.0, cost)   # ignore overlapping non-bonded contacts
    return cost


def cleavage_curve(atoms, n, i, j, D, cost, d, nbins=400):
    """gamma(s) in kJ/mol per A^2 ... returned in J/m^2. n = unit normal (Cartesian). d = plane spacing."""
    p = atoms.get_positions() @ n
    pa = p[i]; pb = pa + D @ n
    lo = np.minimum(pa, pb); hi = np.maximum(pa, pb)
    s_grid = (np.arange(nbins) + 0.5) / nbins * d
    E = np.zeros(nbins)
    # count planes s + k d in (lo, hi): floor((hi - s)/d) - floor((lo - s)/d)
    for b, s in enumerate(s_grid):
        c = np.floor((hi - s) / d) - np.floor((lo - s) / d)
        E[b] = np.sum(cost * c)
    V = abs(np.linalg.det(atoms.cell[:]))
    gamma = 0.5 * (d / V) * E                    # kcal/mol per A^2
    return s_grid, gamma * KCAL * 1000 / 6.02214076e23 * 1e20  # J/m^2


def surface_energy(atoms, hkl, pairs=None, **kw):
    """Minimum over termination offsets of the cleavage energy of the plane family (hkl) in J/m^2."""
    if pairs is None:
        i, j, d, D, bonded, excl = pair_list(atoms, **kw)
        cost = pair_energies(atoms, i, j, d, bonded, excl)
        pairs = (i, j, D, cost)
    i, j, D, cost = pairs
    B = np.linalg.inv(atoms.cell[:]).T
    g = np.array(hkl, float) @ B
    dsp = 1.0 / np.linalg.norm(g); n = g * dsp
    s, gam = cleavage_curve(atoms, n, i, j, D, cost, dsp)
    return float(gam.min()), float(s[gam.argmin()]), dsp, gam

In [ ]:
%%writefile mofmorph/hybrid.py
"""Face energies for ANY crystal system (not only cubic) and the BFDH / Wulff / hybrid shapes.

Every symmetry family of faces (from faces.families) gets its own UFF cleavage energy, computed
on the SAME standardized cell that the Miller indices refer to, so no cubic assumption is made.
"""
import numpy as np
from ase import Atoms
from ase.build import make_supercell
from .structure import load
from .faces import enumerate_faces, families

from .forcefield import pair_list, pair_energies, surface_energy


def family_names(fams):
    """Plain label like {111}; if two different families share a label (non-cubic cells) add the
    representative direction, e.g. {100}[0,0,1]."""
    cnt = {}
    for f in fams:
        cnt[f["label"]] = cnt.get(f["label"], 0) + 1
    return [f["label"] if cnt[f["label"]] == 1 else f"{f['label']}[{','.join(map(str, f['family']))}]" for f in fams]


def atoms_for_energy(path, c):
    """All atoms of the file (hydrogens included) in a cell whose basis is the conventional basis
    that c's Miller indices refer to.

    structure.load() may find the symmetry only after dropping H (ZIF-8 is such a case) and then
    returns an H-free crystal. Energies must still see every atom, so the file's own atoms are
    re-expressed in the conventional basis (an integer supercell of the file cell). If that is not
    possible (the file holds a supercell of the conventional cell) the crystal's own atoms are used
    and a warning is printed."""
    import warnings
    import spglib
    from ase.io import read
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        at = read(path)
    if c.used_hydrogen or "H" not in at.get_chemical_symbols():
        return Atoms(numbers=c.numbers, scaled_positions=c.frac, cell=c.lattice, pbc=True)
    noh = at[[s != "H" for s in at.get_chemical_symbols()]]
    d = spglib.get_symmetry_dataset((noh.cell[:], noh.get_scaled_positions(), noh.numbers), symprec=c.symprec)
    P = np.array(d.transformation_matrix, float)
    Lc = (at.cell[:].T @ np.linalg.inv(P)).T            # conventional basis in the file's frame
    M = Lc @ np.linalg.inv(at.cell[:])
    if not np.allclose(M, np.rint(M), atol=1e-4):
        print(f"  warning: {path}: conventional cell is not an integer multiple of the file cell; "
              "energies use the H-free structure")
        return Atoms(numbers=c.numbers, scaled_positions=c.frac, cell=c.lattice, pbc=True)
    big = make_supercell(at, np.rint(M).astype(int))
    big.wrap()
    return big


def uff_gammas(path, name, n_families=24, max_index=6, min_cell=None, rcut=12.0, verbose=True):
    """Returns (faces, fams, names, gamma) with gamma[name] in J/m^2 for the n_families widest-spaced families."""
    c = load(path, name)
    F = enumerate_faces(c, max_index)
    fams = families(F)[:n_families]
    names = family_names(fams)
    atoms = atoms_for_energy(path, c)
    i, j, d, D, bonded, excl = pair_list(atoms, rcut=rcut)
    cost = pair_energies(atoms, i, j, d, bonded, excl)
    gamma = {}
    for f, nm in zip(fams, names):
        g = surface_energy(atoms, f["family"], pairs=(i, j, D, cost))[0]
        if verbose:
            print(f"  {nm:18s} d={f['d']:7.3f}  gamma={g:.4f} J/m2")
        if g > 0:
            gamma[nm] = float(g)
    return F, fams, names, gamma, c


def rates(fams, names, gamma, w):
    """Hybrid rate per family: Rb^(1-w) * Rw^w (w=0 BFDH, w=1 Wulff), both scaled to min = 1."""
    byname = dict(zip(names, fams))
    use = [k for k in names if k in gamma]
    dj = {k: byname[k]["d_eff"] for k in use}
    Rb = {k: max(dj.values()) / dj[k] for k in use}; m = min(Rb.values()); Rb = {k: v / m for k, v in Rb.items()}
    g0 = min(gamma[k] for k in use); Rw = {k: gamma[k] / g0 for k in use}
    return {k: Rb[k] ** (1 - w) * Rw[k] ** w for k in use}


def hybrid_shape(F, fams, names, gamma, w):
    """Polytope with one rate per family (keyed by family tuple, so no label collisions)."""
    from .shape import Shape
    from .models import _planes
    R = rates(fams, names, gamma, w)
    rate_by_family = {f["family"]: R[nm] for f, nm in zip(fams, names) if nm in R}
    normals, dists, tags = _planes(F, rate_by_family, set(rate_by_family))
    return Shape(normals, dists, tags)

In [ ]:
%%writefile mofmorph/render.py
import numpy as np
import matplotlib
pass  # inline display in notebooks
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

COLORS = {"{111}+": "#E45756", "{111}-": "#F4A6A6", "{211}+": "#B279A2", "{211}-": "#D9B8D0", "{100}": "#4C78A8", "{111}": "#E45756", "{110}": "#54A24B", "{211}": "#B279A2",
          "{310}": "#F2CF5B", "{311}": "#9D755D", "{221}": "#FF9DA6", "rough": "#D9D9D9"}


def draw(ax, shape, title=None, view=(22, 35)):
    for f in shape.faces:
        c = COLORS.get(f["tag"], "#BAB0AC")
        lw = 0.0 if f["tag"] == "rough" else 0.6
        ax.add_collection3d(Poly3DCollection([f["poly"]], facecolor=c, edgecolor="k", linewidth=lw, alpha=0.95))
    r = np.abs(shape.V).max()
    for s in (ax.set_xlim, ax.set_ylim, ax.set_zlim):
        s(-r, r)
    ax.set_box_aspect((1, 1, 1)); ax.view_init(*view); ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=9)


def row(shapes, titles, path, caption=None, size=2.5):
    fig = plt.figure(figsize=(size * len(shapes), size + 0.6))
    for i, (s, t) in enumerate(zip(shapes, titles)):
        draw(fig.add_subplot(1, len(shapes), i + 1, projection="3d"), s, t)
    if caption:
        fig.text(0.5, 0.01, caption, ha="center", fontsize=8.5)
    plt.savefig(path, dpi=200, bbox_inches="tight"); plt.close(fig)

## 2. Choose a MOF
Upload your CIF (Colab) or put the file next to the notebook. **Any crystal system works** (cubic, tetragonal, hexagonal, low symmetry...). The file must contain the unit cell, so use `.cif` (or an `.xyz`/`.extxyz` that includes the lattice). A plain `.xyz` with no cell cannot give crystal faces and is rejected with a message.

In [ ]:
CIF = 'MOF5.cif'       # change this to your file name (.cif or extxyz with a cell)
NAME = 'MOF-5'
N_FAMILIES = 24        # how many face families to use (more = safer shape, slower)
if not os.path.exists(CIF):
    try:
        from google.colab import files
        up = files.upload(); CIF = list(up)[0]
    except ImportError:
        raise SystemExit('Put the file next to this notebook and set CIF above.')

## 3. Faces and UFF face energies
Every symmetry family of faces gets its own energy, computed on the same cell the Miller indices refer to.

In [ ]:
import sys, warnings, numpy as np
warnings.filterwarnings('ignore'); sys.path.insert(0, '.')
from ase.io import read
from mofmorph.hybrid import uff_gammas, hybrid_shape
from mofmorph.models import describe, bfdh

_a = read(CIF)
if abs(np.linalg.det(_a.cell[:])) < 1e-6:
    raise SystemExit('STOP: this file has no unit cell (no lattice). Crystal faces need a periodic structure; use a CIF.')
F, fams, names, gamma_uff, crystal = uff_gammas(CIF, NAME, n_families=N_FAMILIES)
print(NAME, crystal.spacegroup, f'(#{crystal.sg_number})', len(crystal.numbers), 'atoms in the standard cell')
for n in crystal.notes: print('note:', n)

## 4. External energies (optional)
Paste CHGNet / DFT surface energies in J/m2 here, using the same face names as printed above. Leave it empty to use the UFF values.

In [ ]:
gamma_external = {}      # e.g. {'{100}': 0.12, '{111}': 0.20}
gamma = gamma_external if gamma_external else gamma_uff
print('Using', 'EXTERNAL' if gamma_external else 'UFF', 'energies for', len(gamma), 'face families')

## 5. BFDH, Wulff and hybrid shapes

In [ ]:
import matplotlib.pyplot as plt
from mofmorph import render
ws = [0, 0.25, 0.5, 0.75, 1]
shapes, titles, table = [], [], []
for w in ws:
    try:
        s = hybrid_shape(F, fams, names, gamma, w)
    except Exception as e:
        raise SystemExit(f'Could not close the shape at w={w} (not enough face families). Increase N_FAMILIES and rerun. ({str(e)[:80]})')
    shapes.append(s); nm = describe(s, 0.01); titles.append(f"w={w}\n{nm}")
    table.append((w, nm, round(float(s.summary()["sphericity"]), 3)))
fig = plt.figure(figsize=(3*len(ws), 3.4))
for n, (s, t) in enumerate(zip(shapes, titles)):
    render.draw(fig.add_subplot(1, len(ws), n+1, projection="3d"), s, t)
plt.suptitle(f"{NAME}: w=0 is BFDH, w=1 is Wulff, in between is the hybrid"); plt.show()
for r in table: print(r)
print("BFDH with all faces (40 families):", describe(bfdh(F, 40)[0], 0.01))

## What the result means, and its limits
- If BFDH and Wulff give the **same** shape, the hybrid adds nothing. If they **differ**, the hybrid is in between, and only **experimental crystal shapes** (solvent, temperature, modulator) can say which is right.
- The UFF energies are unrelaxed, with no electrostatics, no solvent and no temperature. Treat them as a baseline to beat.
- Shapes are for the crystal as grown (outside shape), not the unit cell.
- Face names: in non-cubic cells the same plain label can belong to different families, so a direction like `[0,0,1]` is added to the name.
- Tested on MOF-5, HKUST-1, UiO-66, ZIF-8 (cubic) and hMOF-0 (R3m). Very large cells (many hundreds of atoms) can be slow.